# CCTB-Net 双平台计算效率评测 (Google Colab T4 / L4 vs. Edge GTX 1650)

**使用步骤（全程无需手动上传大文件）：**
1. 在 Colab 顶部菜单选择 **代码执行程序 (Runtime) -> 更改运行时类型 (Change runtime type)**，硬件加速器选择 **T4 GPU** 或 **L4 GPU**。
2. 依次运行下方 4 个单元格，直接从 GitHub 拉取 `CCTB-Net` 仓库并完成评测、生成双平台对比表与对比图，最后自动下载结果包 `colab_efficiency_results.zip`。

In [ ]:
# 1. 检查 Colab 分配的 GPU 型号（确保是 Tesla T4 或 NVIDIA L4）
!nvidia-smi

In [ ]:
# 2. 从 GitHub 克隆/更新项目并安装依赖
import os
from pathlib import Path

repo_dir = Path('/content/CCTB-Net')
if not repo_dir.exists():
    !git clone https://github.com/ciang77/CCTB-Net.git /content/CCTB-Net
else:
    !git -C /content/CCTB-Net pull

os.chdir(repo_dir)
print('当前工作目录:', os.getcwd())
!pip install -q numba pyyaml scikit-learn pyarrow

In [ ]:
# 3. 运行 plot_efficiency.py（自动完成 T4/L4 吞吐量与显存评测，并与仓库内自带的 GTX 1650 基线生成双平台对比表与对比图）
!python -m evaluation.plot_efficiency

In [ ]:
# 4. 打印生成的双平台对比 Markdown 表格，并打包下载所有结果回本地
from pathlib import Path
import zipfile
from google.colab import files

md_file = Path('outputs/summary_tables/efficiency_dual_platform.md')
if md_file.exists():
    print(md_file.read_text(encoding='utf-8'))

out_zip = Path('/content/colab_efficiency_results.zip')
with zipfile.ZipFile(out_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in ('outputs/summary_tables', 'outputs/figures'):
        for p in Path(folder).glob('*'):
            if p.is_file():
                zf.write(p, arcname=str(p))

print('打包完成，开始下载:', out_zip)
files.download(str(out_zip))